# PDHG Single Run With a Tunable Theorem Tail (Colab)

This notebook launches **one PDHG run** and replaces the last \(K_{\rm tail}\) iterations with the paper-style tail. If \(s=K-K_{\rm tail}\), \(t=k-s\), and

\[
a_t=\left(\frac{c}{c+t}\right)^{1/2},
\]

the tail used here is

\[
\sigma_{s+t}=\sigma_s a_t,\qquad
\tau_{s+t}=\tau_0 a_t^2,\qquad
\rho_{s+t}=\sigma_s a_t^{1+q}.
\]

The notebook exposes \(K\), \(K_{\rm tail}\), \(c\), \(q\), \(\tau_0\), and either \(\sigma_s\) or \(\lambda_{\rm eff}\). It translates the paper parameter \(q\) to the implementation key

\[
\text{theorem1\_tail\_rho\_power}=1+q.
\]

Thus the proposed special case is obtained with \(c=10\) and \(q=1\), while the submitted-paper exponent is obtained with \(q=1.1\). The phase-retrieval dual-step continuation \(\gamma_k\) is configured separately because it is not part of the \((\sigma_k,\tau_k,\rho_k)\) tail.

## Parameterization choices

Choose one tail parameterization in the settings cell:

- **sigma_s**: enter \(\tau_0\) and \(\sigma_s\); the notebook reports \(\lambda_{\rm eff}=\sigma_s^2/\tau_0\).
- **lambda**: enter \(\tau_0\) and \(\lambda_{\rm eff}\); the notebook derives \(\sigma_s=\sqrt{\lambda_{\rm eff}\tau_0}\).
- **prefix**: keep the chosen prefix endpoint and let the actual value at the switch define \(\sigma_s\).

For all three choices, the notebook ends the scheduler prefix exactly at iteration \(s\), so its endpoint is the requested or selected \(\sigma_s\). Use **poly-1** for a prefix that is linear in iteration number. The existing repo default, **poly-7**, is also available.

Suggested comparisons:

- Submitted-style global exponent: \(q=1.1\).
- Proposed special case: \(q=1\), \(c=10\).
- Paper task-specific \(c\): normally 10, 15 for box inpainting, and 30 for phase retrieval.
- Setting \(\lambda_{\rm eff}\) only renames the tuning unless it is fixed by an independent rule.

In [ ]:
#@title Project and run settings

SETUP_MODE = "git"  #@param ["git", "drive_zip"]
REPO_URL = "https://github.com/Seif-Hussein/dyscode.git"  #@param {type:"string"}
REPO_BRANCH = "codex-pdhg-colab-light-100"  #@param {type:"string"}
DRIVE_ZIP_PATH = "/content/drive/MyDrive/mycode2.zip"  #@param {type:"string"}

REPO_DIR = "/content/mycode2"  #@param {type:"string"}
PYTHON_BIN = "/usr/bin/python3"  #@param {type:"string"}
DRIVE_EXPORT_DIR = "/content/drive/MyDrive/pdhg_theorem_tail_single_run_exports"  #@param {type:"string"}
DRIVE_FFHQ_DATA_DIR = "/content/drive/MyDrive/mycode/test-ffhq"  #@param {type:"string"}
SESSION_TAG = ""  #@param {type:"string"}
RUN_NAME = "PDHG_Theorem_Tail_Single_Run"  #@param {type:"string"}
CONFIG_NAME = "default_ffhq.yaml"  #@param {type:"string"}

INVERSE_TASK = "phase_retrieval"  #@param ["phase_retrieval", "inpainting", "inpainting_rand", "motion_blur", "gaussian_blur", "down_sampling", "down_sampling_explicit", "hdr", "nonlinear_blur", "compression_quantization"]
SEED = 99  #@param {type:"integer"}
TOTAL_IMAGES = 100  #@param {type:"integer"}
BATCH_SIZE = 100  #@param {type:"integer"}
DATA_START_IDX = 0  #@param {type:"integer"}
MEASUREMENT_SIGMA = 0.05  #@param {type:"number"}

# Master prefix and tail.
MAX_ITER = 500  #@param {type:"integer"}
SIGMA_MAX = 10.0  #@param {type:"number"}
PREFIX_TIMESTEP = "poly-1"  #@param ["poly-1", "poly-7"]
PREFIX_SIGMA_MIN = 0.05  #@param {type:"number"}
TAIL_STEPS = 10  #@param {type:"integer"}
TAIL_C = 10.0  #@param {type:"number"}
TAIL_Q = 1.1  #@param {type:"number"}
TAIL_RHO_SCALE = 1.0  #@param {type:"number"}

# Select sigma_s, lambda, or prefix.
TAIL_PARAMETERIZATION = "sigma_s"  #@param ["sigma_s", "lambda", "prefix"]
TAU0 = 0.01  #@param {type:"number"}
SIGMA_S = 0.1  #@param {type:"number"}
LAMBDA_EFF = 1.0  #@param {type:"number"}

# Phase-retrieval gamma_k (the implementation calls this sigma_dual).
GAMMA0 = 1600.0  #@param {type:"number"}
GAMMA_SCHEDULE_MODE = "constant"  #@param ["constant", "to_zero", "to_infinity"]
GAMMA_SCHEDULE_SCOPE = "tail"  #@param ["tail", "full"]
GAMMA_SCHEDULE_POWER = 1.0  #@param {type:"number"}
GAMMA_SCHEDULE_MIN = 1e-08  #@param {type:"number"}
GAMMA_SCHEDULE_MAX = 0.0  #@param {type:"number"}

DENOISER_AC_NOISE = True  #@param {type:"boolean"}
DENOISE_FINAL_STEP = "tweedie"  #@param ["tweedie", "ode"]
PHASE_DUAL_ACTIVE_Y_EPS = 0.001  #@param {type:"number"}
PHASE_PR_ALPHA = 0.25  #@param {type:"number"}
EVAL_METRICS = "psnr;ssim;lpips"  #@param {type:"string"}
SAVE_SAMPLES = False  #@param {type:"boolean"}
SAVE_TRAJ = False  #@param {type:"boolean"}
SAVE_TRAJ_RAW_DATA = False  #@param {type:"boolean"}
PROGRESS_JSON_EVERY = 1  #@param {type:"integer"}
LOG_TAIL_LINES = 120  #@param {type:"integer"}

# Optional Hydra overrides separated by semicolons.
# Box inpainting example:
# inverse_task.operator.mask_type=box;inverse_task.operator.mask_len_range=[128,129]
EXTRA_HYDRA_OVERRIDES = ""  #@param {type:"string"}

In [ ]:
#@title Preview and validate the exact schedule
import json
import math

def build_schedule_plan():
    K = int(MAX_ITER)
    tail_steps = int(TAIL_STEPS)
    tau0 = float(TAU0)
    sigma_max = float(SIGMA_MAX)
    prefix_sigma_min = float(PREFIX_SIGMA_MIN)
    c = float(TAIL_C)
    q = float(TAIL_Q)
    rho_scale = float(TAIL_RHO_SCALE)

    if K < 3:
        raise ValueError("MAX_ITER must be at least 3.")
    if tail_steps < 2 or tail_steps >= K:
        raise ValueError("TAIL_STEPS must lie in [2, MAX_ITER - 1].")
    if tau0 <= 0 or sigma_max <= 0 or c <= 0 or rho_scale <= 0:
        raise ValueError("TAU0, SIGMA_MAX, TAIL_C, and TAIL_RHO_SCALE must be positive.")
    if not PREFIX_TIMESTEP.startswith("poly-"):
        raise ValueError("This notebook supports poly-n prefix timesteps only.")
    prefix_power = int(PREFIX_TIMESTEP.split("-", 1)[1])
    if prefix_power <= 0:
        raise ValueError("The poly-n prefix power must be positive.")

    switch_index = K - tail_steps
    scheduler_num_steps = switch_index + 1

    mode = str(TAIL_PARAMETERIZATION).strip().lower()
    if mode == "sigma_s":
        sigma_target = float(SIGMA_S)
        if sigma_target <= 0:
            raise ValueError("SIGMA_S must be positive.")
        lambda_eff = sigma_target ** 2 / tau0
        explicit_lambda = None
    elif mode == "lambda":
        lambda_eff = float(LAMBDA_EFF)
        if lambda_eff <= 0:
            raise ValueError("LAMBDA_EFF must be positive.")
        sigma_target = math.sqrt(lambda_eff * tau0)
        explicit_lambda = lambda_eff
    elif mode == "prefix":
        sigma_target = None
        lambda_eff = None
        explicit_lambda = None
    else:
        raise ValueError("TAIL_PARAMETERIZATION must be sigma_s, lambda, or prefix.")

    if sigma_target is not None:
        scheduler_sigma_min = sigma_target
    else:
        scheduler_sigma_min = prefix_sigma_min
        if scheduler_sigma_min <= 0 or scheduler_sigma_min >= sigma_max:
            raise ValueError("PREFIX_SIGMA_MIN must lie strictly between 0 and SIGMA_MAX.")

    if scheduler_sigma_min <= 0 or scheduler_sigma_min >= sigma_max:
        raise ValueError("sigma_s must lie strictly between 0 and SIGMA_MAX.")

    def sigma_at(index, sigma_min):
        # The repo scheduler gets switch_index + 1 entries. It reaches
        # sigma_min exactly at k=s and is clamped afterward until the tail replaces it.
        clamped_index = min(index, switch_index)
        r = clamped_index / float(switch_index)
        hi = sigma_max ** (1.0 / prefix_power)
        lo = sigma_min ** (1.0 / prefix_power)
        return ((1.0 - r) * hi + r * lo) ** prefix_power

    base_sigma = [sigma_at(k, scheduler_sigma_min) for k in range(K)]
    sigma_switch = base_sigma[switch_index]
    if sigma_target is not None and not math.isclose(sigma_switch, sigma_target, rel_tol=1e-10, abs_tol=1e-12):
        raise RuntimeError("Internal sigma_s targeting check failed.")

    if lambda_eff is None:
        lambda_eff = sigma_switch ** 2 / tau0

    sigma = list(base_sigma)
    tau = [tau0] * K
    rho = list(base_sigma)
    tail_mask = [False] * K
    tail_rows = []
    rho_power_for_code = 1.0 + q

    for k in range(switch_index, K):
        t = k - switch_index
        a = math.sqrt(c / (c + t))
        sigma_k = sigma_switch * a
        tau_k = sigma_k ** 2 / explicit_lambda if explicit_lambda is not None else tau0 * a ** 2
        rho_k = rho_scale * sigma_switch * a ** (1.0 + q)
        sigma[k] = sigma_k
        tau[k] = tau_k
        rho[k] = rho_k
        tail_mask[k] = True
        tail_rows.append({
            "k": k,
            "t": t,
            "a_t": a,
            "sigma": sigma_k,
            "tau": tau_k,
            "rho": rho_k,
        })

    gamma = [float(GAMMA0)] * K
    gamma_mode = str(GAMMA_SCHEDULE_MODE)
    if gamma_mode != "constant":
        if gamma_mode not in {"to_zero", "to_infinity"}:
            raise ValueError("Unsupported GAMMA_SCHEDULE_MODE.")
        anchor_index = switch_index if GAMMA_SCHEDULE_SCOPE == "tail" else 0
        anchor_sigma = sigma[anchor_index]
        gamma_power = float(GAMMA_SCHEDULE_POWER)
        gamma_min = float(GAMMA_SCHEDULE_MIN) if float(GAMMA_SCHEDULE_MIN) > 0 else None
        gamma_max = float(GAMMA_SCHEDULE_MAX) if float(GAMMA_SCHEDULE_MAX) > 0 else None
        for k in range(anchor_index, K):
            ratio = sigma[k] / anchor_sigma
            value = float(GAMMA0) * (ratio ** gamma_power if gamma_mode == "to_zero" else ratio ** (-gamma_power))
            if gamma_min is not None:
                value = max(value, gamma_min)
            if gamma_max is not None:
                value = min(value, gamma_max)
            gamma[k] = value

    return {
        "K": K,
        "tail_steps": tail_steps,
        "switch_index": switch_index,
        "tail_t_values": [0, tail_steps - 1],
        "prefix_timestep": PREFIX_TIMESTEP,
        "scheduler_num_steps": scheduler_num_steps,
        "scheduler_sigma_min": scheduler_sigma_min,
        "sigma_s": sigma_switch,
        "tau0": tau0,
        "lambda_eff": lambda_eff,
        "explicit_tail_lambda": explicit_lambda,
        "c": c,
        "q": q,
        "rho_scale": rho_scale,
        "rho_power_for_code": rho_power_for_code,
        "sigma": sigma,
        "tau": tau,
        "rho": rho,
        "gamma": gamma,
        "tail_mask": tail_mask,
        "tail_rows": tail_rows,
    }

schedule_plan = build_schedule_plan()

summary = {
    key: schedule_plan[key]
    for key in [
        "K", "tail_steps", "switch_index", "tail_t_values", "prefix_timestep",
        "scheduler_num_steps", "scheduler_sigma_min", "sigma_s", "tau0", "lambda_eff",
        "explicit_tail_lambda", "c", "q", "rho_scale", "rho_power_for_code",
    ]
}
print(json.dumps(summary, indent=2))
print()
print("First three tail rows:")
for row in schedule_plan["tail_rows"][:3]:
    print(row)
print()
print("Last three tail rows:")
for row in schedule_plan["tail_rows"][-3:]:
    print(row)

try:
    import matplotlib.pyplot as plt
    k_values = list(range(schedule_plan["K"]))
    fig, axes = plt.subplots(2, 2, figsize=(12, 7), constrained_layout=True)
    for ax, key, label in [
        (axes[0, 0], "sigma", "sigma_k"),
        (axes[0, 1], "tau", "tau_k"),
        (axes[1, 0], "rho", "rho_k"),
        (axes[1, 1], "gamma", "gamma_k"),
    ]:
        ax.plot(k_values, schedule_plan[key])
        ax.axvline(schedule_plan["switch_index"], color="tab:red", linestyle="--", alpha=0.7)
        ax.set_title(label)
        ax.set_xlabel("iteration k")
        ax.grid(alpha=0.25)
    plt.show()
except ImportError:
    print("matplotlib is unavailable; skipping the plot.")

In [ ]:
#@title Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
#@title Fetch the repository
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

repo_dir = Path(REPO_DIR)
repo_dir.parent.mkdir(parents=True, exist_ok=True)
os.chdir(repo_dir.parent)

if repo_dir.exists():
    shutil.rmtree(repo_dir)

if SETUP_MODE == "git":
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, repo_dir.as_posix()],
        check=True,
    )
elif SETUP_MODE == "drive_zip":
    zip_path = Path(DRIVE_ZIP_PATH)
    if not zip_path.exists():
        raise FileNotFoundError(f"Zip file not found: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(repo_dir.parent)
    extracted_root = repo_dir.parent / zip_path.stem
    if extracted_root.exists() and extracted_root != repo_dir:
        if repo_dir.exists():
            shutil.rmtree(repo_dir)
        extracted_root.rename(repo_dir)
else:
    raise ValueError(f"Unsupported SETUP_MODE: {SETUP_MODE}")

os.chdir(repo_dir)
print(f"Repository ready: {repo_dir}")

In [ ]:
#@title Install Colab-safe dependencies
import os
import subprocess

os.chdir(REPO_DIR)
subprocess.run([PYTHON_BIN, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
print("Installed requirements-colab.txt")

In [ ]:
#@title Download the FFHQ checkpoint if needed
import os
import subprocess
from pathlib import Path

os.chdir(REPO_DIR)
checkpoint_path = Path("pretrained-models/ffhq_10m.pt")
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

if checkpoint_path.exists():
    print(f"Checkpoint already present: {checkpoint_path}")
else:
    subprocess.run(
        ["gdown", "--id", "1BGwhRWUoguF-D8wlZ65tf227gp3cDUDh", "-O", checkpoint_path.as_posix()],
        check=True,
    )
    print(f"Downloaded checkpoint to: {checkpoint_path}")

In [ ]:
#@title Build the single-run command
import json
import os
import shlex
import time
from pathlib import Path

os.chdir(REPO_DIR)
repo_dir = Path(REPO_DIR)
drive_data_dir = Path(DRIVE_FFHQ_DATA_DIR)
if not drive_data_dir.exists():
    raise FileNotFoundError(f"FFHQ dataset path not found: {drive_data_dir}")

def parse_semicolon_list(text):
    return [
        item.strip()
        for item in str(text).replace("\n", ";").split(";")
        if item.strip()
    ]

metric_list = parse_semicolon_list(EVAL_METRICS)
if not metric_list:
    raise ValueError("EVAL_METRICS must contain at least one metric.")
extra_overrides = parse_semicolon_list(EXTRA_HYDRA_OVERRIDES)

# Recompute so changed settings cannot accidentally reuse a stale preview.
schedule_plan = build_schedule_plan()
session_tag = SESSION_TAG.strip() or time.strftime("%Y%m%d-%H%M%S")
run_tag = f"pdhg_theorem_tail_{INVERSE_TASK}_{session_tag}"
run_name = f"{RUN_NAME}_{session_tag}"
save_root = repo_dir / "results" / "single_runs" / run_tag
hydra_root = repo_dir / "outputs" / "single_runs" / run_tag
run_aux_root = repo_dir / "single_runs"
latest_log_path = run_aux_root / f"{run_tag}.log"
latest_pid_path = run_aux_root / f"{run_tag}.pid"
progress_path = run_aux_root / f"{run_tag}.progress.json"
context_path = run_aux_root / f"{run_tag}.context.json"
schedule_manifest_path = run_aux_root / f"{run_tag}.schedule.json"
data_end_idx = int(DATA_START_IDX) + int(TOTAL_IMAGES)

run_cmd = [
    PYTHON_BIN,
    "recover_inverse2.py",
    "--config-name",
    CONFIG_NAME,
    "sampler=edm_pdhg",
    f"inverse_task={INVERSE_TASK}",
    f"name={run_name}",
    f"seed={int(SEED)}",
    "gpu=0",
    "wandb=false",
    "show_config=false",
    f"save_samples={'true' if SAVE_SAMPLES else 'false'}",
    f"save_traj={'true' if SAVE_TRAJ else 'false'}",
    f"save_traj_raw_data={'true' if SAVE_TRAJ_RAW_DATA else 'false'}",
    f"total_images={int(TOTAL_IMAGES)}",
    f"batch_size={int(BATCH_SIZE)}",
    "num_runs=1",
    f"inverse_task.operator.sigma={float(MEASUREMENT_SIGMA)}",
    f"sampler.annealing_scheduler_config.num_steps={schedule_plan['scheduler_num_steps']}",
    f"sampler.annealing_scheduler_config.sigma_max={float(SIGMA_MAX)}",
    f"sampler.annealing_scheduler_config.sigma_min={schedule_plan['scheduler_sigma_min']}",
    "sampler.annealing_scheduler_config.schedule=linear",
    f"sampler.annealing_scheduler_config.timestep={PREFIX_TIMESTEP}",
    f"++sampler.annealing_scheduler_config.theorem1_tail_steps={schedule_plan['tail_steps']}",
    "++sampler.annealing_scheduler_config.theorem1_tail_sigma_mode=theorem1",
    f"++sampler.annealing_scheduler_config.theorem1_tail_c={schedule_plan['c']}",
    f"++sampler.annealing_scheduler_config.theorem1_tail_rho_power={schedule_plan['rho_power_for_code']}",
    f"++sampler.annealing_scheduler_config.theorem1_tail_rho_scale={schedule_plan['rho_scale']}",
    f"inverse_task.admm_config.max_iter={schedule_plan['K']}",
    f"++inverse_task.admm_config.early_stop={schedule_plan['K']}",
    f"++inverse_task.admm_config.pdhg.tau={schedule_plan['tau0']}",
    f"++inverse_task.admm_config.pdhg.sigma_dual={float(GAMMA0)}",
    f"++inverse_task.admm_config.pdhg.sigma_dual_schedule_mode={GAMMA_SCHEDULE_MODE}",
    f"++inverse_task.admm_config.pdhg.sigma_dual_schedule_scope={GAMMA_SCHEDULE_SCOPE}",
    "++inverse_task.admm_config.pdhg.sigma_dual_schedule_tail_steps=0",
    f"++inverse_task.admm_config.pdhg.sigma_dual_schedule_power={float(GAMMA_SCHEDULE_POWER)}",
    f"inverse_task.admm_config.denoise.final_step={DENOISE_FINAL_STEP}",
    f"++inverse_task.admm_config.denoise.ac_noise={'true' if DENOISER_AC_NOISE else 'false'}",
    "inverse_task.admm_config.denoise.lgvd.num_steps=0",
    f"++inverse_task.admm_config.pdhg.phase_dual_active_y_eps={float(PHASE_DUAL_ACTIVE_Y_EPS)}",
    f"++inverse_task.admm_config.pdhg.phase_pr_alpha={float(PHASE_PR_ALPHA)}",
    f"eval_fn_list=[{','.join(metric_list)}]",
    f"data.image_root_path={drive_data_dir.as_posix()}",
    f"data.start_idx={int(DATA_START_IDX)}",
    f"data.end_idx={data_end_idx}",
    f"++progress_json_path={progress_path.as_posix()}",
    f"++progress_json_every={max(1, int(PROGRESS_JSON_EVERY))}",
]

if schedule_plan["explicit_tail_lambda"] is not None:
    run_cmd.append(
        f"++sampler.annealing_scheduler_config.theorem1_tail_lambda={schedule_plan['explicit_tail_lambda']}"
    )
if float(GAMMA_SCHEDULE_MIN) > 0:
    run_cmd.append(
        f"++inverse_task.admm_config.pdhg.sigma_dual_schedule_min={float(GAMMA_SCHEDULE_MIN)}"
    )
if float(GAMMA_SCHEDULE_MAX) > 0:
    run_cmd.append(
        f"++inverse_task.admm_config.pdhg.sigma_dual_schedule_max={float(GAMMA_SCHEDULE_MAX)}"
    )

run_cmd.extend(extra_overrides)
run_cmd.extend([
    f"save_dir={save_root.as_posix()}",
    f"hydra.run.dir={hydra_root.as_posix()}",
])

run_aux_root.mkdir(parents=True, exist_ok=True)
manifest = {
    "formulas": {
        "a_t": "sqrt(c / (c + t))",
        "sigma": "sigma_s * a_t",
        "tau": "tau0 * a_t^2",
        "rho": "rho_scale * sigma_s * a_t^(1 + q)",
    },
    **schedule_plan,
}
schedule_manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

last_context = {
    "run_tag": run_tag,
    "run_name": run_name,
    "save_root": save_root.as_posix(),
    "hydra_root": hydra_root.as_posix(),
    "latest_log_path": latest_log_path.as_posix(),
    "latest_pid_path": latest_pid_path.as_posix(),
    "run_progress_path": progress_path.as_posix(),
    "context_path": context_path.as_posix(),
    "schedule_manifest_path": schedule_manifest_path.as_posix(),
    "run_cmd": run_cmd,
}
context_path.write_text(json.dumps(last_context, indent=2), encoding="utf-8")

print(f"Run tag: {run_tag}")
print(f"Task: {INVERSE_TASK}")
print(f"Dataset slice: [{DATA_START_IDX}, {data_end_idx})")
print(f"K: {schedule_plan['K']}")
print(f"Prefix scheduler entries: {schedule_plan['scheduler_num_steps']}")
print(f"Tail switch s: {schedule_plan['switch_index']}")
print(f"Tail entries: {schedule_plan['tail_steps']} (t=0..{schedule_plan['tail_steps'] - 1})")
print(f"sigma_s: {schedule_plan['sigma_s']:.10g}")
print(f"tau0: {schedule_plan['tau0']:.10g}")
print(f"lambda_eff: {schedule_plan['lambda_eff']:.10g}")
print(f"c: {schedule_plan['c']:.10g}")
print(f"q: {schedule_plan['q']:.10g}")
print(f"implementation rho power: {schedule_plan['rho_power_for_code']:.10g}")
print(f"gamma mode: {GAMMA_SCHEDULE_MODE}")
print(f"Save root: {save_root}")
print(f"Schedule manifest: {schedule_manifest_path}")
if extra_overrides:
    print(f"Extra overrides: {extra_overrides}")
print("\nCommand:\n")
print(" ".join(shlex.quote(part) for part in run_cmd))

In [ ]:
#@title Validate the Hydra configuration without running the model
import subprocess

validation_cmd = last_context["run_cmd"][:4] + ["--cfg", "job"] + last_context["run_cmd"][4:]
validation = subprocess.run(
    validation_cmd,
    cwd=REPO_DIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(validation.stdout[-12000:])
if validation.returncode != 0:
    raise RuntimeError(f"Hydra validation failed with exit code {validation.returncode}")
print("Hydra configuration is valid.")

In [ ]:
#@title Launch the run in the background
import os
import subprocess
from pathlib import Path

os.chdir(REPO_DIR)
log_path = Path(last_context["latest_log_path"])
pid_path = Path(last_context["latest_pid_path"])
log_path.parent.mkdir(parents=True, exist_ok=True)

with log_path.open("w", encoding="utf-8") as log_handle:
    process = subprocess.Popen(
        last_context["run_cmd"],
        cwd=REPO_DIR,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        text=True,
    )

pid_path.write_text(str(process.pid), encoding="utf-8")
print(f"PID: {process.pid}")
print(f"Log: {log_path}")
print(f"Progress: {last_context['run_progress_path']}")
print(f"Save root: {last_context['save_root']}")

In [ ]:
#@title Show recent log lines
from pathlib import Path

log_path = Path(last_context["latest_log_path"])
if not log_path.exists():
    raise FileNotFoundError(f"Log file not found: {log_path}")

lines = log_path.read_text(encoding="utf-8", errors="ignore").splitlines()
print("\n".join(lines[-int(LOG_TAIL_LINES):]) if lines else "<log is empty>")

In [ ]:
#@title Show live progress and current tail values
import json
from pathlib import Path

progress_path = Path(last_context["run_progress_path"])
if not progress_path.exists():
    raise FileNotFoundError(f"Progress file not found yet: {progress_path}")

payload = json.loads(progress_path.read_text(encoding="utf-8"))
print(json.dumps(payload, indent=2))

step = payload.get("step")
if isinstance(step, int) and 1 <= step <= schedule_plan["K"]:
    index = step - 1
    print()
    print("Schedule at the latest completed iteration:")
    print({
        "k": index,
        "tail_active": schedule_plan["tail_mask"][index],
        "sigma": schedule_plan["sigma"][index],
        "tau": schedule_plan["tau"][index],
        "rho": schedule_plan["rho"][index],
        "gamma": schedule_plan["gamma"][index],
    })

In [ ]:
#@title Show results and recorded schedule tails
import json
from pathlib import Path

save_root = Path(last_context["save_root"])
metrics_matches = sorted(save_root.rglob("metrics.json"))
history_matches = sorted(save_root.rglob("metric_history.json"))
eval_matches = sorted(save_root.rglob("eval.md"))
grid_matches = sorted(save_root.rglob("grid_results.png"))

print(f"save_root: {save_root}")
print(f"metrics files: {[p.as_posix() for p in metrics_matches]}")
print(f"history files: {[p.as_posix() for p in history_matches]}")
print(f"evaluation files: {[p.as_posix() for p in eval_matches]}")
print(f"image grids: {[p.as_posix() for p in grid_matches]}")

if metrics_matches:
    print()
    print("Final metrics:")
    print(json.dumps(json.loads(metrics_matches[0].read_text(encoding="utf-8")), indent=2))

if history_matches:
    history = json.loads(history_matches[0].read_text(encoding="utf-8"))
    history_view = history.get("runs", [history])[0] if isinstance(history, dict) else {}
    print()
    print("Recorded final schedule values:")
    for key in ["sigma", "tau", "rho", "sigma_dual"]:
        values = history_view.get(key, []) if isinstance(history_view, dict) else []
        if values:
            print(f"{key}: len={len(values)} tail={values[-min(10, len(values)):]}")

if eval_matches:
    print()
    print(eval_matches[0].read_text(encoding="utf-8", errors="ignore"))

In [ ]:
#@title Copy run artifacts to Google Drive
import shutil
from pathlib import Path

export_root = Path(DRIVE_EXPORT_DIR)
export_root.mkdir(parents=True, exist_ok=True)

targets = [
    Path(last_context["save_root"]),
    Path(last_context["hydra_root"]),
    Path(last_context["latest_log_path"]),
    Path(last_context["context_path"]),
    Path(last_context["schedule_manifest_path"]),
    Path(last_context["run_progress_path"]),
]

for source in targets:
    if not source.exists():
        print(f"Skipping missing path: {source}")
        continue
    destination = export_root / source.name
    if source.is_dir():
        if destination.exists():
            shutil.rmtree(destination)
        shutil.copytree(source, destination)
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
    print(f"Copied {source} -> {destination}")